# Column Type & Format Suggester

**Contents:**
1. Setup & data loading
2. Load Model & Data
3. Per-type precision
4. Macro-F1 and overall accuracy
5. Type confusion matrix
6. Format accuracy per temporal / IP type
7. Feature importance chart
8. Confidence calibration analysis

## 1. Setup

In [ ]:
import sys, os

_here = os.path.abspath('')
_root = _here
for _ in range(3):
    if os.path.isdir(os.path.join(_root, 'src')):
        break
    _root = os.path.dirname(_root)
os.chdir(_root)
sys.path.insert(0, os.path.join(_root, 'src'))
print(f'Project root: {_root}')

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.model_selection import train_test_split

from src import config
from src import features as F
from src.type_format_suggester import suggest, detect_format

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Setup complete')


## 2. Load Model & Data

In [ ]:
artifact = joblib.load('model.joblib')
model = artifact['model']
le = artifact['label_encoder']
model_type = artifact['model_type']

print(f'Model type: {model_type}')
print(f'Test macro-F1 (from training): {artifact["test_macro_f1"]:.4f}')
print(f'Test accuracy (from training): {artifact["test_accuracy"]:.4f}')
print(f'Classes: {list(le.classes_)}')


In [ ]:
cache = np.load('data/features.npz', allow_pickle=True)
X, y_str = cache['x'], cache['y']
y_enc = le.transform(y_str)

i, X_test, j, y_test = train_test_split(
    X, y_enc,
    test_size=0.2,
    random_state=config.RANDOM_SEED,
    stratify=y_enc
)
y_pred = model.predict(X_test)
y_test_labels = le.inverse_transform(y_test)
y_pred_labels = le.inverse_transform(y_pred)

print(f'Test set size: {len(X_test)} columns')
print(f'Classes: {len(le.classes_)}')


## 3. Per-Type Precision


In [ ]:
report = classification_report(
    y_test_labels, y_pred_labels,
    target_names=sorted(le.classes_),
    output_dict=True
)
report_df = pd.DataFrame(report).T.round(3)

def highlight_f1(val):
    if not isinstance(val, float): 
        return ''
    if val < 0.85: 
        return 'background-color: #f8d7da'
    if val < 0.95: 
        return 'background-color: #fff3cd'
    return 'background-color: #d1e7dd'

report_df.style.map(highlight_f1, subset=['f1-score'])


## 4. Macro-F1 & Overall Accuracy

In [ ]:
acc = accuracy_score(y_test_labels, y_pred_labels)
macro_f1 = f1_score(y_test_labels, y_pred_labels, average='macro')
weighted_f1 = f1_score(y_test_labels, y_pred_labels, average='weighted')

print(f'Accuracy: {acc:.4f}  ({acc*100:.1f}%)')
print(f'Macro-F1: {macro_f1:.4f}')
print(f'Weighted-F1: {weighted_f1:.4f}')

types = sorted(le.classes_)
f1s   = [float(report[t]['f1-score']) for t in types] # type: ignore

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#d1e7dd' if f >= 0.95 else '#fff3cd' if f >= 0.85 else '#f8d7da' for f in f1s]
bars = ax.barh(types, f1s, color=colors, edgecolor='#333', linewidth=0.5)

ax.axvline(float(macro_f1), color='#0d6efd', linestyle='--', linewidth=1.5, label=f'Macro-F1 = {macro_f1:.3f}')

ax.set_xlim(0, 1.05)
ax.set_xlabel('F1-Score')
ax.set_title('Per-Type F1 Score', fontweight='bold')
ax.legend()
for bar, val in zip(bars, f1s):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
    
plt.tight_layout()
plt.show()


## 5. Type Confusion Matrix

In [ ]:
labels = sorted(le.classes_)
cm = confusion_matrix(y_test_labels, y_pred_labels, labels=labels)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax, shrink=0.8)

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title('Type Confusion Matrix', fontsize=13, fontweight='bold')

thresh = cm.max() / 2
for i in range(len(labels)):
    for j in range(len(labels)):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8, color='white' if cm[i, j] > thresh else 'black')

plt.tight_layout()
plt.show()

print('Misclassifications:')
from collections import Counter
errs = Counter((t, p) for t, p in zip(y_test_labels, y_pred_labels) if t != p)
for (t, p), n in errs.most_common():
    print(f'  {t:<16} -> {p:<16} {n}')
if not errs:
    print('None — perfect classification on test set.')


## 6. Format Accuracy per Temporal / IP Type

In [ ]:
dataset = json.load(open('data/labeled_columns.json'))

format_types = list(config.TYPES_REQUIRING_FORMAT)
results = {t: {'correct': 0, 'total': 0, 'ambiguous': 0} for t in format_types}

for col_data in dataset:
    t = col_data['type']
    if t not in format_types:
        continue
    
    true_fmt = col_data['format']
    vals = pd.Series([v for v in col_data['values'] if v and str(v).strip()])
    
    if len(vals) == 0:
        continue
    
    pred_fmt, resolved = detect_format(t, vals)
    results[t]['total'] += 1
    
    if not resolved:
        results[t]['ambiguous'] += 1
    if pred_fmt == true_fmt:
        results[t]['correct'] += 1

print(f'{'Type':<16} {'Correct':>8} {'Total':>7} {'Accuracy':>10} {'Ambiguous':>11}')
print('-' * 58)
for t, r in results.items():
    acc = r['correct'] / r['total'] if r['total'] else 0
    print(f"{t:<16} {r['correct']:>8} {r['total']:>7} {acc:>9.1%} {r['ambiguous']:>11}")


## 7. Feature Importance Chart

In [ ]:
importances = model.feature_importances_
feat_names  = artifact['feature_names']

imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=True)

def feat_color(name):
    if name.startswith('ratio_'):  
        return '#0d6efd'
    if name.startswith('name_'):   
        return '#fd7e14'
    return '#198754'

colors = [feat_color(n) for n in imp_df['feature']]

fig, ax = plt.subplots(figsize=(9, 9))
bars = ax.barh(imp_df['feature'], imp_df['importance'], color=colors, edgecolor='#333', linewidth=0.4)

ax.set_xlabel('Mean Decrease in Impurity (feature importance)')
ax.set_title(f'Feature Importance — {model_type}', fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#0d6efd', label='Regex match-ratio'),
    Patch(facecolor='#198754', label='Statistical'),
    Patch(facecolor='#fd7e14', label='Column-name signal'),
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

print('Top 5 most important features:')
print(imp_df.sort_values('importance', ascending=False).head(5).to_string(index=False))


## 8. Confidence Calibration Analysis

In [ ]:
proba = model.predict_proba(X_test)
max_conf = proba.max(axis=1)
true_labels = le.inverse_transform(y_test)

ambiguous_types = {config.TYPE_TRUE_FALSE, config.TYPE_WHOLE_NUMBER}
clean_types = {config.TYPE_EMAIL, config.TYPE_IP_ADDRESS, config.TYPE_DATE, config.TYPE_DATE_TIME, config.TYPE_TIME}

conf_amb = [c for c, l in zip(max_conf, true_labels) if l in ambiguous_types]
conf_clean = [c for c, l in zip(max_conf, true_labels) if l in clean_types]

print(f'Ambiguous samples found: {len(conf_amb)}')
print(f'Ambiguous types (bool/int) with mean confidence: {np.mean(conf_amb):.3f}\n')
print(f'Clean samples found: {len(conf_clean)}')
print(f'Clean types (email/ip/date) with mean confidence: {np.mean(conf_clean):.3f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(conf_amb, bins=50, color='#dc3545', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Model confidence')
ax1.set_ylabel('Count')
ax1.set_title('Ambiguous Types (bool/int)', fontweight='bold')
ax1.axvline(np.mean(conf_amb), color='black', linestyle='--', linewidth=2, label=f'Mean: {np.mean(conf_amb):.3f}')
ax1.legend()

ax2.hist(conf_clean, bins=50, color='#198754', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Model confidence')
ax2.set_ylabel('Count')
ax2.set_title('Clean Types (email/IP/date)', fontweight='bold')
ax2.axvline(np.mean(conf_clean), color='black', linestyle='--', linewidth=2, label=f'Mean: {np.mean(conf_clean):.3f}')
ax2.legend()

plt.tight_layout()
plt.show()